# ### Sales Intelligence & Revenue Forecasting Platform

A company has sales data from customers, products, orders, sales representatives, regions, and marketing activities. The business wants to understand what happened, why it happened, what will happen next, and which customers are likely to buy or churn.

Sales / CRM / ERP / Excel / APIs
              ↓
        Data Ingestion
   ADF / Auto Loader / APIs
              ↓
       Bronze Delta Layer
      Raw Sales Transactions
              ↓
        Silver Layer
   Clean + Transform + Join
              ↓
        Gold Layer
 Customer / Product / Sales KPIs
              ↓
        AI/ML Feature Layer
              ↓
 ┌───────────────────────────────┐
 │ 1. Sales Forecasting          │
 │ 2. Customer Churn Prediction  │
 │ 3. Customer Purchase Propensity│
 │ 4. Sales Opportunity Scoring  │
 │ 5. Product Recommendation     │
 └───────────────────────────────┘
              ↓
        MLflow Tracking
              ↓
     Predictions / Scores
              ↓
       Gold AI Tables
              ↓
       Power BI Dashboard
              ↓
      Business Decisions

In [0]:
# ============================================================
# SALES INTELLIGENCE PLATFORM
# End-to-End Databricks AI/ML Project
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    RegressionEvaluator
)

import random
from datetime import datetime, timedelta

spark = SparkSession.builder.getOrCreate()

# Database
spark.sql("CREATE DATABASE IF NOT EXISTS sales_ai")
spark.sql("USE sales_ai")

print("Sales AI project initialized")

In [0]:
# ============================================================
# PRODUCT DATA
# ============================================================

categories = [
    "Electronics",
    "Furniture",
    "Clothing",
    "Software",
    "Office Supplies"
]

products = []

for i in range(1, 101):

    category = random.choice(categories)

    price_raw = random.uniform(100, 50000)
    price = float(f"{price_raw:.2f}")
    
    cost_raw = random.uniform(50, 30000)
    cost = float(f"{cost_raw:.2f}")
    
    products.append((
        f"PROD{i:04d}",
        f"Product_{i}",
        category,
        price,
        cost
    ))

product_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("cost", DoubleType(), True)
])

product_df = spark.createDataFrame(
    products,
    product_schema
)

product_df.write.format("delta").mode("overwrite").saveAsTable(
    "products_bronze"
)

display(product_df)

In [0]:
# ============================================================
# CUSTOMER DATA
# ============================================================

random.seed(42)

num_customers = 10000

regions = ["North", "South", "East", "West"]
segments = ["Consumer", "Corporate", "Small Business"]
cities = [
    "Delhi", "Mumbai", "Bangalore", "Hyderabad",
    "Chennai", "Pune", "Kolkata", "Ahmedabad"
]

customers = []

for i in range(1, num_customers + 1):

    customers.append((
        f"CUST{i:06d}",
        random.randint(18, 70),
        random.choice(regions),
        random.choice(segments),
        random.choice(cities),
        datetime(2022, 1, 1) + timedelta(
            days=random.randint(0, 1000)
        )
    ))

customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("age", IntegerType(), True),
    StructField("region", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("city", StringType(), True),
    StructField("signup_date", DateType(), True)
])

customer_df = spark.createDataFrame(
    customers,
    customer_schema
)

customer_df.write.format("delta").mode("overwrite").saveAsTable(
    "customers_bronze"
)

display(customer_df.limit(10))

In [0]:
# ============================================================
# SALES TRANSACTIONS
# ============================================================

num_sales = 500000

sales = []

start_date = datetime(2023, 1, 1)

for i in range(1, num_sales + 1):

    customer_id = f"CUST{random.randint(1, num_customers):06d}"
    product_id = f"PROD{random.randint(1, 100):04d}"

    quantity = random.randint(1, 10)

    order_date = start_date + timedelta(
        days=random.randint(0, 1095)
    )

    unit_price_raw = random.uniform(100, 50000)
    unit_price = float(f"{unit_price_raw:.2f}")

    discount_raw = random.uniform(0, 0.30)
    discount = float(f"{discount_raw:.2f}")

    sales_amount_raw = quantity * unit_price * (1 - discount)
    sales_amount = float(f"{sales_amount_raw:.2f}")

    sales.append((
        f"ORD{i:08d}",
        customer_id,
        product_id,
        order_date.date(),
        quantity,
        unit_price,
        discount,
        sales_amount
    ))

sales_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("order_date", DateType(), False),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount", DoubleType(), True),
    StructField("sales_amount", DoubleType(), True)
])

sales_df = spark.createDataFrame(
    sales,
    sales_schema
)

sales_df.write.format("delta").mode("overwrite").saveAsTable(
    "sales_transactions_bronze"
)

print(f"Generated {sales_df.count()} sales transactions")

display(sales_df.limit(10))

In [0]:
# ============================================================
# MARKETING DATA
# ============================================================

campaigns = [
    "Email",
    "Google Ads",
    "Social Media",
    "SMS",
    "Referral"
]

marketing = []

for i in range(1, 10001):

    customer_id = f"CUST{i:06d}"

    emails = random.randint(0, 50)
    clicks = random.randint(0, emails)
    website_visits = random.randint(0, 100)
    conversions = random.randint(0, clicks if clicks < 10 else 10)

    marketing.append((
        customer_id,
        random.choice(campaigns),
        emails,
        clicks,
        website_visits,
        conversions
    ))

marketing_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("campaign", StringType(), True),
    StructField("emails_sent", IntegerType(), True),
    StructField("email_clicks", IntegerType(), True),
    StructField("website_visits", IntegerType(), True),
    StructField("conversions", IntegerType(), True)
])

marketing_df = spark.createDataFrame(
    marketing,
    marketing_schema
)

marketing_df.write.format("delta").mode("overwrite").saveAsTable(
    "marketing_bronze"
)

display(marketing_df.limit(10))

In [0]:
# ============================================================
# SILVER SALES
# ============================================================

sales_bronze = spark.table("sales_transactions_bronze")

sales_silver = (
    sales_bronze

    # Remove duplicates
    .dropDuplicates(["order_id"])

    # Valid transactions
    .filter(col("sales_amount") > 0)
    .filter(col("quantity") > 0)

    # Derived fields
    .withColumn(
        "year",
        year("order_date")
    )

    .withColumn(
        "month",
        month("order_date")
    )

    .withColumn(
        "month_start",
        trunc("order_date", "month")
    )

    .withColumn(
        "profit",
        col("sales_amount") -
        (col("quantity") * col("unit_price") * 0.60)
    )
)

sales_silver.write.format("delta").mode("overwrite").saveAsTable(
    "sales_silver"
)

display(sales_silver.limit(10))

In [0]:
# ============================================================
# CUSTOMER 360
# ============================================================

customer_360 = (
    sales_silver
    .groupBy("customer_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),

        sum("sales_amount").alias("total_revenue"),

        avg("sales_amount").alias("avg_order_value"),

        sum("quantity").alias("total_quantity"),

        avg("discount").alias("avg_discount"),

        sum("profit").alias("total_profit"),

        max("order_date").alias("last_purchase_date"),

        min("order_date").alias("first_purchase_date")
    )
)

# Recency
customer_360 = customer_360.withColumn(
    "days_since_last_purchase",
    datediff(
        current_date(),
        col("last_purchase_date")
    )
)

customer_360.write.format("delta").mode("overwrite").saveAsTable(
    "customer_360_gold"
)

display(customer_360.limit(20))

In [0]:
# ============================================================
# CUSTOMER 360 ENRICHMENT
# ============================================================

customers = spark.table("customers_bronze")

customer_360_final = (
    customer_360
    .join(
        customers,
        "customer_id",
        "left"
    )
)

customer_360_final.write.format("delta").mode("overwrite").saveAsTable(
    "customer_360_final_gold"
)

display(customer_360_final.limit(20))

In [0]:
# ============================================================
# SALES PERFORMANCE
# ============================================================

sales_performance = (
    sales_silver
    .join(
        customers.select(
            "customer_id",
            "region",
            "segment"
        ),
        "customer_id",
        "left"
    )
    .groupBy(
        "year",
        "month",
        "region",
        "segment"
    )
    .agg(
        sum("sales_amount").alias("total_sales"),
        sum("profit").alias("total_profit"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers"),
        sum("quantity").alias("units_sold")
    )
)

sales_performance.write.format("delta").mode("overwrite").saveAsTable(
    "sales_performance_gold"
)

display(sales_performance)

In [0]:
# ============================================================
# ML FEATURE ENGINEERING
# ============================================================

features = (
    customer_360_final

    .withColumn(
        "avg_orders_per_month",
        col("total_orders") / 36
    )

    .withColumn(
        "revenue_per_order",
        col("total_revenue") /
        when(col("total_orders") == 0, 1)
        .otherwise(col("total_orders"))
    )

    .withColumn(
        "customer_age",
        col("age")
    )

    .withColumn(
        "high_value_customer",
        when(
            col("total_revenue") > 200000,
            1
        ).otherwise(0)
    )
)

display(features.limit(20))